# MT01：期中實作考每人數據產生器（以學號為種子）

**食品分析實驗 W9 期中實作考｜教師用**

開書、可討論的實作考，最關鍵的一步是**每個人拿到的數據都不一樣，但要做的判斷一樣**。這份筆記本用學號當亂數種子，為每位學生產生兩份長格式 CSV：

| 卷 | 情境 | 學生要做的事 | 對應的筆記本／練習檔 |
|---|---|---|---|
| A 卷 | 醋的可滴定酸：手動滴定 vs PASCO 自動滴定，8 個樣品 | KHP 標定 → 算 % 酸 → 差值的描述統計 → Grubbs 找異常樣品 → 配對 t 與 Bland–Altman → 3 句結論 | NB04 / EX04 |
| B 卷 | 全班能力試驗：5 組盲樣灰分、12 點脂肪 QC 管制圖、蛋白質放行判定 | 單因子 ANOVA → z-score 分級 → Westgard 規則 → 以擴充不確定度 U 對照規格做放行判定 | NB08 / EX08 |

每位學生的數據裡都埋了同一類「設計好的狀況」，但**位置、大小、方向因人而異**，所以同組可以討論「該用哪個檢定、門檻怎麼查」，卻沒辦法直接抄數字：

- A 卷：手動滴定有一個小而顯著的正偏差（酚酞終點略過當量點），另外有一個樣品的手動讀數明顯讀錯（滴定管讀差 1 mL）。
- B 卷：5 組中有一組灰分偏高（灰化不完全），z-score 落在 $|z| \ge 3$；管制圖裡有一個 Westgard 違規（$1_{3s}$ 或 $2_{2s}$ 二選一）；蛋白質放行判定的結果在「放行／不放行／無法判斷」三種之中隨機。

程式會**反覆抽樣直到每位學生的數據都符合設計**（偏差顯著、異常值 Grubbs 判得出來、ANOVA 顯著、z-score 分級正確、管制圖只違反設計的那一條規則），所以答案鍵裡的結論對每個人都成立，閱卷時可以用同一份規準。

## 怎麼用

1. 在第 1 節貼上學生名冊（學號，一行一個；也可以上傳 CSV）。
2. 設定 `EXAM_SALT`（例如 `"2026-W9"`）。同一個學號加同一個 salt 永遠產生同一份數據；明年換 salt，整批數據就不一樣。
3. 「全部執行」。輸出一個 zip（每人兩份 CSV）和教師用的 `answer_key.csv`／`answer_key.xlsx`。
4. 第 6 節有可以直接貼到考卷上的題目說明；第 7 節可以預覽任一位學生的 Bland–Altman 圖與管制圖，確認埋的狀況看得出來。

> 數據是模擬的，參數取自 NB04 與 NB08 的教學資料（醋約 4–6% 醋酸、奶粉灰分約 6%、脂肪 QC 約 26%）。所有圖表的座標軸一律用英文，避免 Colab 預設字型出現方框。

## 資料格式

沿用全課程共用的長格式 `group, student, item, rep, value, unit, temp_C, note, sample`。考卷數據的 `note` 欄一律留空，不給任何提示。

## 1. 環境設定與學生名冊 (Setup & Roster)

In [ ]:
import hashlib
import io
import os
import zipfile

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

# ---- 考試設定 ---------------------------------------------------------
EXAM_SALT = "2026-W9"        # 每學期改一次；同學號 + 同 salt = 同一份數據
OUTPUT_DIR = "midterm_data"  # 輸出資料夾（Colab 的工作目錄下）
ZIP_NAME = "midterm_student_data.zip"

名冊可以用兩種方式提供：

1. 直接把學號貼在下面的 `ROSTER_TEXT`，一行一個；若要在檔名裡帶組別，寫成 `學號,組別`（例如 `B1130001,G1`）。
2. 上傳一個 CSV（欄位 `student_id`，可選 `group`），執行下一個 cell 中被註解掉的上傳程式碼。

下面預設 15 個示範學號（5 組、每組 3 人），正式使用前請換成你的名冊。

In [ ]:
ROSTER_TEXT = '''
B1130001,G1
B1130002,G1
B1130003,G1
B1130004,G2
B1130005,G2
B1130006,G2
B1130007,G3
B1130008,G3
B1130009,G3
B1130010,G4
B1130011,G4
B1130012,G4
B1130013,G5
B1130014,G5
B1130015,G5
'''


def parse_roster(text):
    rows = []
    for line in text.strip().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = [p.strip() for p in line.replace("\t", ",").split(",")]
        sid = parts[0]
        grp = parts[1] if len(parts) > 1 and parts[1] else ""
        rows.append({"student_id": sid, "group": grp})
    df = pd.DataFrame(rows)
    if df["student_id"].duplicated().any():
        dup = df.loc[df["student_id"].duplicated(), "student_id"].tolist()
        raise ValueError(f"名冊有重複學號：{dup}")
    return df


roster = parse_roster(ROSTER_TEXT)
print(f"名冊共 {len(roster)} 人")
roster.head()

In [ ]:
# （選用）在 Colab 上傳名冊 CSV（欄位 student_id，可選 group）。非 Colab 環境執行不會有作用。
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     roster = pd.read_csv(fname, dtype=str).fillna("")
#     roster.columns = [c.strip().lower() for c in roster.columns]
#     if "group" not in roster.columns:
#         roster["group"] = ""
#     roster = roster[["student_id", "group"]]
#     print(f"已改用上傳名冊：{fname}，共 {len(roster)} 人")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟。")

## 2. 學號 → 亂數種子 (Seed from Student ID)

用 SHA-256 把「學號 + salt」雜湊成一個整數當種子。這樣：

- 同一個學號永遠拿到同一份數據，考後補發、重算答案都不會對不上。
- 相鄰學號（B1130001 與 B1130002）的數據完全不相關，不會出現「數字只差一點」的情況。
- 種子裡不含任何可以反推學號的資訊。

In [ ]:
def seed_from_id(student_id, salt=EXAM_SALT):
    h = hashlib.sha256(f"{salt}|{student_id}".encode("utf-8")).hexdigest()
    return int(h[:8], 16)  # 32-bit 整數


def grubbs_crit(n, alpha=0.05):
    # 兩側 Grubbs 臨界值（與 NB08 相同的公式）
    t = stats.t.ppf(1 - alpha / (2 * n), n - 2)
    return (n - 1) / np.sqrt(n) * np.sqrt(t ** 2 / (n - 2 + t ** 2))


def grubbs_stat(x):
    x = np.asarray(x, dtype=float)
    return float(np.max(np.abs(x - x.mean())) / x.std(ddof=1))


print({sid: seed_from_id(sid) for sid in roster["student_id"].head(3)})

## 3. A 卷：手動 vs 自動滴定 (Method Comparison)

**模擬模型**（與 NB04 相同的公式）：

- NaOH 標定：$N = \dfrac{m_{KHP}}{204.22 \times V_L}$，每人 3 次重複，標稱濃度約 0.1 N（每人略有不同）。
- 可滴定酸（以醋酸計）：$\%酸 = \dfrac{N \times V \times 60.05}{W \times 1000} \times 100$。
- 8 個醋樣品的真值分布在 3.0–6.0%，PASCO 直接回報 % 酸（隨機誤差 SD 0.03%）。
- 手動滴定有一個正偏差 $\delta$（每人 0.06–0.10%，酚酞變色略晚於當量點），滴定管讀數四捨五入到 0.05 mL。
- 其中一個樣品（位置隨機）的手動讀數多讀了 1.0 mL（讀錯刻度），換算後約 +0.3%。

**設計保證**（不符合就換一組噪音重抽）：

1. 含異常樣品時，8 個差值的 Grubbs $G > G_{crit}(n=8)$。
2. 排除異常樣品後，7 對配對 t 檢定 $p < 0.01$（偏差顯著）。
3. 排除後 Bland–Altman 的 95% 一致性界限不包含 0（偏差不是隨機）。

In [ ]:
MW_KHP = 204.22
EQ_WT_ACETIC = 60.05
N_SAMPLES_A = 8


def generate_A(student_id, group=""):
    base_seed = seed_from_id(student_id)
    for attempt in range(200):
        rng = np.random.default_rng(base_seed + attempt * 7919)

        # --- NaOH 標定（KHP 三重複）
        N_true = rng.uniform(0.0980, 0.1040)
        khp_mass = np.round(rng.uniform(0.49, 0.53, 3), 4)
        khp_vol = np.round(khp_mass / (MW_KHP * N_true) * 1000 + rng.normal(0, 0.03, 3), 2)
        N_est = float(np.mean(khp_mass / (MW_KHP * khp_vol / 1000)))

        # --- 8 個醋樣品
        true_pct = np.sort(rng.uniform(3.0, 6.0, N_SAMPLES_A))
        sample_mass = np.round(rng.uniform(1.95, 2.10, N_SAMPLES_A), 4)
        bias_manual = rng.uniform(0.06, 0.10)
        pasco_pct = np.round(true_pct + rng.normal(0, 0.03, N_SAMPLES_A), 3)
        manual_pct_target = true_pct + bias_manual + rng.normal(0, 0.03, N_SAMPLES_A)
        manual_vol = manual_pct_target / 100 * sample_mass * 1000 / (N_true * EQ_WT_ACETIC)
        outlier_idx = int(rng.integers(0, N_SAMPLES_A))
        manual_vol[outlier_idx] += 1.0  # 讀錯 1 mL
        manual_vol = np.round(np.round(manual_vol / 0.05) * 0.05, 2)

        # --- 答案鍵計算（用學生會算出的 N_est，而非 N_true）
        manual_pct = N_est * manual_vol * EQ_WT_ACETIC / (sample_mass * 1000) * 100
        d_all = manual_pct - pasco_pct
        G = grubbs_stat(d_all)
        Gc = grubbs_crit(N_SAMPLES_A)
        G_idx = int(np.argmax(np.abs(d_all - d_all.mean())))
        keep = np.ones(N_SAMPLES_A, bool)
        keep[outlier_idx] = False
        d = d_all[keep]
        t_stat, p_val = stats.ttest_rel(manual_pct[keep], pasco_pct[keep])
        loa_lo = d.mean() - 1.96 * d.std(ddof=1)
        loa_hi = d.mean() + 1.96 * d.std(ddof=1)
        r_all = float(np.corrcoef(manual_pct, pasco_pct)[0, 1])

        ok = (G > Gc) and (G_idx == outlier_idx) and (p_val < 0.01) and (loa_lo > 0)
        if ok:
            break
    else:
        raise RuntimeError(f"{student_id}: A 卷 200 次內無法滿足設計條件")

    # --- 長格式輸出
    rows = []
    for i in range(3):
        rows.append([group, student_id, "khp_mass_g", i + 1, khp_mass[i], "g", 25.0, "", "KHP"])
        rows.append([group, student_id, "khp_titration_mL", i + 1, khp_vol[i], "mL", 25.0, "", "NaOH"])
    for i in range(N_SAMPLES_A):
        s = f"vinegar_{i + 1:02d}"
        rows.append([group, student_id, "sample_mass_g", i + 1, sample_mass[i], "g", 25.0, "", s])
        rows.append([group, student_id, "manual_titration_mL", i + 1, manual_vol[i], "mL", 25.0, "", s])
        rows.append([group, student_id, "pasco_pct_acid", i + 1, pasco_pct[i], "%", 25.0, "", s])
    data = pd.DataFrame(rows, columns=["group", "student", "item", "rep", "value",
                                       "unit", "temp_C", "note", "sample"])

    key = {
        "A_attempt": attempt,
        "A_NaOH_N_true": round(N_true, 5),
        "A_NaOH_N_est": round(N_est, 5),
        "A_bias_manual_design_%": round(bias_manual, 3),
        "A_outlier_sample": f"vinegar_{outlier_idx + 1:02d}",
        "A_outlier_manual_mL_error": 1.0,
        "A_r_all8": round(r_all, 4),
        "A_diff_mean_all8_%": round(float(d_all.mean()), 4),
        "A_diff_sd_all8_%": round(float(d_all.std(ddof=1)), 4),
        "A_grubbs_G": round(G, 3),
        "A_grubbs_Gcrit_n8": round(Gc, 3),
        "A_diff_mean_excl_%": round(float(d.mean()), 4),
        "A_diff_sd_excl_%": round(float(d.std(ddof=1)), 4),
        "A_paired_t_excl": round(float(t_stat), 3),
        "A_paired_p_excl": f"{p_val:.2e}",
        "A_LoA_low_%": round(float(loa_lo), 4),
        "A_LoA_high_%": round(float(loa_hi), 4),
        "A_conclusion": "手動比自動系統性偏高，排除讀錯的樣品後偏差仍顯著；兩法不可互換，需校正終點或改用 pH 8.2",
    }
    return data, key


demo_A, demo_keyA = generate_A(roster.loc[0, "student_id"], roster.loc[0, "group"])
print(pd.Series(demo_keyA))
demo_A.head(12)

## 4. B 卷：能力試驗、管制圖、放行判定 (Proficiency, Control Chart, Release)

**模擬模型**（參數與 NB08 相同）：

- 盲樣（全脂奶粉）灰分真值約 6.0%，5 組各 3 重複，組內 SD 0.03%。其中一組（隨機）灰化不完全，偏高 0.32–0.40%。
- z-score 以 5 組平均的**中位數**為指定值、$\sigma_{pt} = 0.10\%$（固定），分級 $|z|\le2$ 滿意、$2<|z|<3$ 可疑、$|z|\ge3$ 不滿意。
- 脂肪 QC 樣品 12 次（平均 26.0%、SD 0.15%），埋一個 Westgard 違規：$1_{3s}$（單點超過 ±3s）或 $2_{2s}$（連續兩點同側超過 2s），位置在第 4–12 點之間隨機。
- 蛋白質放行：規格 ≥ 25.0%，報告值 ± U（k = 2，U = 0.4%）；結果在三種情況之間隨機：放行（區間全在規格內）、不放行（區間全在規格外）、無法判斷（區間跨過規格限）。

**設計保證**：

1. 單因子 ANOVA $p < 0.01$。
2. 偏高那一組 $|z| \ge 3$，其他四組 $|z| < 2$。
3. 管制圖只違反設計的那一條規則，其他四條（$R_{4s}$、$4_{1s}$、$10_x$ 等）都不觸發。

In [ ]:
GROUPS_B = ["G1", "G2", "G3", "G4", "G5"]
SIGMA_PT = 0.10
QC_MEAN, QC_SD, QC_N = 26.0, 0.15, 12
PROT_SPEC, PROT_U = 25.0, 0.4


def westgard(x, mean, sd):
    # 回傳觸發的規則集合（與 NB08 第 6 節同一套定義）
    z = (np.asarray(x) - mean) / sd
    hits = set()
    if np.any(np.abs(z) > 3):
        hits.add("1_3s")
    for i in range(1, len(z)):
        if abs(z[i]) > 2 and abs(z[i - 1]) > 2 and np.sign(z[i]) == np.sign(z[i - 1]):
            hits.add("2_2s")
        if (z[i] > 2 and z[i - 1] < -2) or (z[i] < -2 and z[i - 1] > 2):
            hits.add("R_4s")
    for i in range(3, len(z)):
        w = z[i - 3:i + 1]
        if np.all(w > 1) or np.all(w < -1):
            hits.add("4_1s")
    for i in range(9, len(z)):
        w = z[i - 9:i + 1]
        if np.all(w > 0) or np.all(w < 0):
            hits.add("10_x")
    return hits


def generate_B(student_id, group=""):
    base_seed = seed_from_id(student_id)
    for attempt in range(200):
        rng = np.random.default_rng(base_seed + 104729 + attempt * 7919)

        # --- 灰分能力試驗
        ash_true = rng.uniform(5.95, 6.05)
        biased_idx = int(rng.integers(0, 5))
        bias_ash = rng.uniform(0.32, 0.40)
        ash = {}
        for gi, g in enumerate(GROUPS_B):
            mu = ash_true + rng.normal(0, 0.02) + (bias_ash if gi == biased_idx else 0.0)
            ash[g] = np.round(mu + rng.normal(0, 0.03, 3), 2)
        means = np.array([ash[g].mean() for g in GROUPS_B])
        F, p_anova = stats.f_oneway(*[ash[g] for g in GROUPS_B])
        assigned = float(np.median(means))
        z = (means - assigned) / SIGMA_PT
        z_ok = (abs(z[biased_idx]) >= 3) and all(abs(z[i]) < 2 for i in range(5) if i != biased_idx)

        # --- 脂肪 QC 管制圖
        rule = str(rng.choice(["1_3s", "2_2s"]))
        qc_z = np.clip(rng.normal(0, 0.8, QC_N), -1.5, 1.5)
        qc_z[::2] *= np.sign(qc_z[::2]) * rng.choice([-1, 1], size=len(qc_z[::2]))  # 打散符號，避免 10_x
        sign = rng.choice([-1, 1])
        if rule == "1_3s":
            pos = int(rng.integers(3, QC_N))
            qc_z[pos] = sign * rng.uniform(3.3, 3.8)
            rule_pos = [pos + 1]
        else:
            pos = int(rng.integers(3, QC_N - 1))
            qc_z[pos] = sign * rng.uniform(2.2, 2.7)
            qc_z[pos + 1] = sign * rng.uniform(2.2, 2.7)
            rule_pos = [pos + 1, pos + 2]
        qc = np.round(QC_MEAN + qc_z * QC_SD, 2)
        hits = westgard(qc, QC_MEAN, QC_SD)

        # --- 蛋白質放行
        outcome = str(rng.choice(["release", "reject", "undetermined"]))
        if outcome == "release":
            prot = PROT_SPEC + PROT_U + rng.uniform(0.10, 0.30)
        elif outcome == "reject":
            prot = PROT_SPEC - PROT_U - rng.uniform(0.10, 0.30)
        else:
            prot = PROT_SPEC + rng.uniform(-0.25, 0.25)
        prot = round(float(prot), 2)
        lo, hi = prot - PROT_U, prot + PROT_U
        outcome_check = "release" if lo >= PROT_SPEC else ("reject" if hi < PROT_SPEC else "undetermined")

        ok = (p_anova < 0.01) and z_ok and (hits == {rule}) and (outcome_check == outcome)
        if ok:
            break
    else:
        raise RuntimeError(f"{student_id}: B 卷 200 次內無法滿足設計條件")

    rows = []
    for g in GROUPS_B:
        for i in range(3):
            rows.append([g, student_id, "ash", i + 1, ash[g][i], "%", "", "", "blind_milk_powder"])
    for i in range(QC_N):
        rows.append(["QC", student_id, "fat_qc_check", i + 1, qc[i], "%", "", "", "fat_QC_material"])
    rows.append(["LAB", student_id, "protein_result", 1, prot, "%", "", "", "blind_milk_powder"])
    rows.append(["LAB", student_id, "protein_U_k2", 1, PROT_U, "%", "", "", "blind_milk_powder"])
    rows.append(["LAB", student_id, "protein_spec_min", 1, PROT_SPEC, "%", "", "", "blind_milk_powder"])
    data = pd.DataFrame(rows, columns=["group", "student", "item", "rep", "value",
                                       "unit", "temp_C", "note", "sample"])

    zh_grade = lambda v: "滿意" if abs(v) <= 2 else ("可疑" if abs(v) < 3 else "不滿意")
    zh_outcome = {"release": "放行", "reject": "不放行", "undetermined": "無法判斷：區間跨過規格限"}[outcome]
    key = {
        "B_attempt": attempt,
        "B_ash_true_%": round(ash_true, 3),
        "B_biased_group": GROUPS_B[biased_idx],
        "B_bias_design_%": round(bias_ash, 3),
        "B_anova_F": round(float(F), 2),
        "B_anova_p": f"{p_anova:.2e}",
        "B_assigned_value_%": round(assigned, 4),
        **{f"B_z_{g}": round(float(z[i]), 2) for i, g in enumerate(GROUPS_B)},
        **{f"B_grade_{g}": zh_grade(z[i]) for i, g in enumerate(GROUPS_B)},
        "B_westgard_rule": rule,
        "B_westgard_points": "+".join(map(str, rule_pos)),
        "B_protein_result_%": prot,
        "B_protein_interval": f"[{lo:.2f}, {hi:.2f}]",
        "B_release_decision": zh_outcome,
    }
    return data, key


demo_B, demo_keyB = generate_B(roster.loc[0, "student_id"], roster.loc[0, "group"])
print(pd.Series(demo_keyB))
demo_B.query("item == 'ash'").pivot(index="rep", columns="group", values="value")

## 5. 批次產生、答案鍵與打包 (Batch Generation & Answer Key)

每人兩個檔案：`<學號>_A_titration.csv`、`<學號>_B_proficiency.csv`。答案鍵一列一人，存成 CSV 與 Excel。

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
key_rows = []
file_list = []

for _, r in roster.iterrows():
    sid, grp = r["student_id"], r["group"]
    dA, kA = generate_A(sid, grp)
    dB, kB = generate_B(sid, grp)
    fA = os.path.join(OUTPUT_DIR, f"{sid}_A_titration.csv")
    fB = os.path.join(OUTPUT_DIR, f"{sid}_B_proficiency.csv")
    dA.to_csv(fA, index=False)
    dB.to_csv(fB, index=False)
    file_list += [fA, fB]
    key_rows.append({"student_id": sid, "group": grp, "seed": seed_from_id(sid), **kA, **kB})

answer_key = pd.DataFrame(key_rows)
key_csv = os.path.join(OUTPUT_DIR, "answer_key.csv")
answer_key.to_csv(key_csv, index=False, encoding="utf-8-sig")
try:
    answer_key.to_excel(os.path.join(OUTPUT_DIR, "answer_key.xlsx"), index=False)
except Exception as e:
    print("Excel 答案鍵未寫出（缺 openpyxl？）：", e)

with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in file_list:
        zf.write(f, arcname=os.path.basename(f))

print(f"已產生 {len(roster)} 人 × 2 份 CSV → {ZIP_NAME}（不含答案鍵）")
print(f"答案鍵：{key_csv}")
answer_key[["student_id", "group", "A_outlier_sample", "A_paired_p_excl",
            "B_biased_group", "B_anova_p", "B_westgard_rule", "B_release_decision"]]

### 確認每個人的結論都如設計

下面的檢查全部要是 `True`。`attempt` 欄告訴你某位學生重抽了幾次才符合設計（通常是 0–3 次）。

In [ ]:
checks = {
    "A: Grubbs 判得出異常樣品": (answer_key["A_grubbs_G"] > answer_key["A_grubbs_Gcrit_n8"]).all(),
    "A: 排除後配對 t p<0.01": (answer_key["A_paired_p_excl"].astype(float) < 0.01).all(),
    "A: 排除後 LoA 下限 > 0": (answer_key["A_LoA_low_%"] > 0).all(),
    "A: 全部 8 點 r > 0.9（r 高但不可互換）": (answer_key["A_r_all8"] > 0.9).all(),
    "B: ANOVA p<0.01": (answer_key["B_anova_p"].astype(float) < 0.01).all(),
    "B: 恰好一組不滿意": (answer_key[[f"B_grade_{g}" for g in GROUPS_B]].eq("不滿意").sum(axis=1) == 1).all(),
    "B: 其他組都滿意": (answer_key[[f"B_grade_{g}" for g in GROUPS_B]].eq("滿意").sum(axis=1) == 4).all(),
    "每人答案鍵都不同（異常樣品/偏高組/規則/放行）": (
        answer_key[["A_outlier_sample", "B_biased_group", "B_westgard_rule", "B_release_decision"]]
        .astype(str).agg("|".join, axis=1).nunique() > 1
    ),
    "同學號重算得到相同數據": generate_A(roster.loc[0, "student_id"], roster.loc[0, "group"])[0].equals(demo_A),
}
for k, v in checks.items():
    print(f"{'OK ' if v else 'NG '} {k}")
print()
print("重抽次數：A 卷 max =", answer_key["A_attempt"].max(), "，B 卷 max =", answer_key["B_attempt"].max())
print("放行判定分布：", answer_key["B_release_decision"].str.slice(0, 4).value_counts().to_dict())
print("Westgard 規則分布：", answer_key["B_westgard_rule"].value_counts().to_dict())

In [ ]:
# （選用）在 Colab 下載 zip 與答案鍵。非 Colab 環境執行不會有作用。
# try:
#     from google.colab import files
#     files.download(ZIP_NAME)
#     files.download(key_csv)
# except ImportError:
#     print("非 Colab 環境，檔案在", os.path.abspath(OUTPUT_DIR))

## 6. 給學生的題目說明（可直接貼到考卷）

下面是和這份數據配套的任務。兩卷都是「判定、依據、下一步」三格；公式與臨界值表另附在考卷上。

In [ ]:
EXAM_TEXT = f'''
期中實作考（開書、同組可討論；每人數據不同，各自交檔）

你拿到兩個 CSV（長格式：group, student, item, rep, value, unit, temp_C, note, sample）。
用 Excel 或 Colab 任選，完成下列任務並交出檔案與一頁結論。每個判定都要寫出引用的數據與門檻。

A 卷　醋的可滴定酸：手動滴定 vs PASCO 自動滴定（8 個醋樣品）
 A1  用 3 次 KHP 標定算出 NaOH 濃度 N（平均、SD、RSD）。
 A2  用 N、滴定體積、樣品重算出 8 個樣品的手動 % 醋酸（當量重 60.05）。PASCO 的 % 酸已直接給在 pasco_pct_acid。
 A3  算 8 個「手動 − 自動」差值的平均與 SD；用 Grubbs 檢定（α = 0.05）判斷有沒有異常樣品，寫出 G 與 G_crit。
 A4  排除異常樣品後，用配對 t 檢定與 Bland–Altman 圖（平均差、95% 一致性界限）比較兩種方法。
 A5  三句結論：兩法有沒有系統偏差？方向與大小？可不可以互換？另外說明為什麼相關係數 r 很高卻不能當作「兩法一致」的證據。

B 卷　全班能力試驗與品管（盲樣奶粉）
 B1  5 組灰分（各 3 重複）：算各組平均與 SD，做單因子 ANOVA，寫出 F、p 與結論。
 B2  以 5 組平均的中位數為指定值、σ_pt = {SIGMA_PT}%，算各組 z-score 並分級（|z| ≤ 2 滿意、2 < |z| < 3 可疑、|z| ≥ 3 不滿意）。
     指出哪一組有問題，並提出一個可能的實驗原因與下一步。
 B3  脂肪 QC 樣品 12 次：以平均 {QC_MEAN}%、SD {QC_SD}% 畫 Shewhart 管制圖，判斷是否違反 Westgard 規則（1_3s、2_2s、R_4s、4_1s、10_x），
     寫出違反的規則與第幾點，以及該批數據要不要放行。
 B4  蛋白質報告值 ± U（k = 2）對照規格 ≥ {PROT_SPEC}%：判定放行、不放行或無法判斷，並說明理由。
     若為「無法判斷」，提出實驗室下一步該做什麼。

交卷前請填：本組討論中我不同意的一點，以及我最後怎麼決定（不計對錯，空白扣分）。
'''
print(EXAM_TEXT)

## 7. 預覽任一位學生的圖（教師檢查用）

把 `PREVIEW_ID` 換成名冊裡的任一學號，看看埋的狀況在圖上是否明顯。

In [ ]:
PREVIEW_ID = roster.loc[0, "student_id"]
dA, kA = generate_A(PREVIEW_ID, "")
dB, kB = generate_B(PREVIEW_ID, "")

# --- A: 學生會做的計算
khp_m = dA.query("item == 'khp_mass_g'").sort_values("rep")["value"].to_numpy()
khp_v = dA.query("item == 'khp_titration_mL'").sort_values("rep")["value"].to_numpy()
N_est = np.mean(khp_m / (MW_KHP * khp_v / 1000))
w = dA.query("item == 'sample_mass_g'").sort_values("rep")["value"].to_numpy()
vm = dA.query("item == 'manual_titration_mL'").sort_values("rep")["value"].to_numpy()
auto = dA.query("item == 'pasco_pct_acid'").sort_values("rep")["value"].to_numpy()
manual = N_est * vm * EQ_WT_ACETIC / (w * 1000) * 100
diff = manual - auto
avg = (manual + auto) / 2

# --- B: 管制圖
qc = dB.query("item == 'fat_qc_check'").sort_values("rep")["value"].to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ax = axes[0]
ax.scatter(auto, manual)
lim = [min(auto.min(), manual.min()) - 0.2, max(auto.max(), manual.max()) + 0.2]
ax.plot(lim, lim, "k--", lw=1)
ax.set_xlabel("PASCO % acetic acid"); ax.set_ylabel("Manual % acetic acid")
ax.set_title(f"Scatter (r = {np.corrcoef(auto, manual)[0,1]:.3f})")

ax = axes[1]
ax.scatter(avg, diff)
ax.axhline(diff.mean(), color="C1", label="mean diff (all 8)")
ax.axhline(0, color="k", lw=1)
for i, s in enumerate(dA.query("item == 'sample_mass_g'").sort_values("rep")["sample"]):
    ax.annotate(s[-2:], (avg[i], diff[i]), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Mean of methods (%)"); ax.set_ylabel("Manual - PASCO (%)")
ax.set_title(f"Bland-Altman; planted outlier = {kA['A_outlier_sample']}")
ax.legend(fontsize=8)

ax = axes[2]
x = np.arange(1, QC_N + 1)
ax.plot(x, qc, "o-")
for k_, ls in [(0, "-"), (1, ":"), (2, "--"), (3, "-.")]:
    for sgn in (1, -1):
        ax.axhline(QC_MEAN + sgn * k_ * QC_SD, color="gray", ls=ls, lw=0.8)
ax.set_xlabel("QC run"); ax.set_ylabel("Fat QC (%)")
ax.set_title(f"Shewhart chart; planted {kB['B_westgard_rule']} at run {kB['B_westgard_points']}")
plt.tight_layout()
plt.show()

print(f"{PREVIEW_ID}: N_est = {N_est:.5f}, Grubbs G = {kA['A_grubbs_G']} vs Gcrit = {kA['A_grubbs_Gcrit_n8']}, "
      f"biased ash group = {kB['B_biased_group']}, release = {kB['B_release_decision']}")

## 8. 閱卷與延伸

- **閱卷**：對照 `answer_key.csv` 的欄位（`A_outlier_sample`、`A_paired_p_excl`、`B_biased_group`、`B_westgard_rule`、`B_release_decision`），再用評量工具包「8_評分規準」評判斷與結論。數字錯但推理對給大部分分數；數字對但沒寫依據只給一半。
- **A、B 卷分開考**：把第 5 節的迴圈改成只呼叫 `generate_A` 或 `generate_B` 即可。
- **調整難度**：想讓異常值更明顯，把 `generate_A` 裡的 `+= 1.0`（mL）加大；想讓 ANOVA 更不明顯，把 `bias_ash` 的範圍縮小。改完重跑第 5 節的檢查，確認設計保證仍然成立。
- **補考或下學期**：換 `EXAM_SALT` 就是全新的一批數據；名冊、題目說明、答案鍵格式都不用動。
- **搭配期末**：W18 的光譜實作考可以用同一套做法，把 NB11 的模擬光譜產生器改成以學號為種子。